# Clustering

## K-Means

Instances which are more similar to each other are given the same _label_. It works by finding centroids of each "blob"'s center and assign the instances to the closest blob. 

- **Hard Clustering**: each instance is assigned to a single cluster. 
- **Soft Clustering**: each instance is assigned a score (it can be distance between instance and centroid; or it can be similarity score such as Gaussian Radial Basis Function (RBF)).

It works by initializing random centroids, generating labels with those centroids, then update the centroids and generate labels, and so on, until the centroids stop moving. The algorithm is guaranteed to converge in a finite number of steps, as mean squared distance between instances and closest centroid can only go down at each step.  

**Inertia**: mean squared distance between each instance and its closest centroid. It is used as a performance metric. 

In Scikit-Learn, it is implemented using `sklearn.cluster.KMeans` class.

### Improvements to K-Means Clustering

1. **K-Means++**: K-Means with a smarter initialization step that tends to select centroids that are distant from one another. This is default method in Scikit-Learn's `KMeans()` class.
2. **Accelerated K-Means**: accelerates algorithm by exploiting the triangle inequality and by keeping track of lower and upper bounds for disrtances between instances and centroids. Used by default in Scikit-Learn.
3. **Mini-batch K-Means**: instead of full dataset at each iteration, it uses mini-batches to move the centroids slightly in each iteration. Can be used with `sklearn.cluster.MiniBatchKMeans` class. 

### Finding the optimal number of clusters

#### Silhouette Score

- Silhouette score is the mean silhouette coefficient over all instances. 
- An instance's silhouette coefficient is $\frac{(b-a)}{\text{max}(a, b)}$ where $a$ is mean distance to other instances in the same cluster and $b$ is the mean distance to the instances of the next closest cluster, defined as the one that minimizes $b$.
- The coefficient varies between -1 to 1.
- Closer to +1 means instance is inside it's own cluster, and far from others.
- Closer to 0 means it's close to a cluster boundary.
- Closer to -1 means it's assigned to the wrong cluster.
- In Scikit-Learn, it is implemented via `sklearn.metrics.silhouette_score` class.

### Limits of K-Means

- It needs to run several time to avoid suboptimal solutions.
- The number of clusters ($k$) must be specified.
- Behaves differently when clusters are of different sizes, densities, or if they are of nonspherical shapes.

### Applications

#### Using Clustering for Image Segmentation

In [6]:
from matplotlib.image import imread
import os

image = imread(os.path.join("images", "ladybug.jpeg"))
image.shape

(452, 679, 3)

In [7]:
X = image.reshape(-1, 3)

In [8]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=8).fit(X)
segmented_img = kmeans.cluster_centers_[kmeans.labels_]
segmented_img = segmented_img.reshape(image.shape)

#### Using Clustering from Preprocessing

In [14]:
from sklearn.datasets import load_digits

X_digits, y_digits = load_digits(return_X_y=True)

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_digits, y_digits)

In [16]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression()
log_reg.fit(X_train, y_train)

/home/haider/anaconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [17]:
log_reg.score(X_test, y_test)

0.9755555555555555

In [18]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("kmeans", KMeans(n_clusters=50)), 
    ("log_reg", LogisticRegression()), 
])
pipeline.fit(X_train, y_train)

/home/haider/anaconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,steps,"[('kmeans', ...), ('log_reg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,n_clusters,50
,init,'k-means++'
,n_init,'auto'
,max_iter,300
,tol,0.0001
,verbose,0
,random_state,None


In [19]:
pipeline.score(X_test, y_test)

0.9666666666666667

In [20]:
from sklearn.model_selection import GridSearchCV

param_grid = dict(kmeans__n_clusters=range(2, 100))
grid_clf = GridSearchCV(pipeline, param_grid, cv=3,  verbose=2)
grid_clf.fit(X_train, y_train)

Fitting 3 folds for each of 98 candidates, totalling 294 fits
[CV] END ...............................kmeans__n_clusters=2; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=2; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=2; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=3; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=3; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=3; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=4; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=4; total time=   0.0s


/home/haider/anaconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/haider/anaconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules

[CV] END ...............................kmeans__n_clusters=4; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=5; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=5; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=5; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=6; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=6; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=6; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=7; total time=   0.0s


/home/haider/anaconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/haider/anaconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules

[CV] END ...............................kmeans__n_clusters=7; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=7; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=8; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=8; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=8; total time=   0.1s


...

[CV] END ..............................kmeans__n_clusters=97; total time=   0.1s
[CV] END ..............................kmeans__n_clusters=97; total time=   0.1s
[CV] END ..............................kmeans__n_clusters=97; total time=   0.1s
[CV] END ..............................kmeans__n_clusters=98; total time=   0.1s


/home/haider/anaconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/haider/anaconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules

[CV] END ..............................kmeans__n_clusters=98; total time=   0.1s
[CV] END ..............................kmeans__n_clusters=98; total time=   0.1s
[CV] END ..............................kmeans__n_clusters=99; total time=   0.1s
[CV] END ..............................kmeans__n_clusters=99; total time=   0.0s
[CV] END ..............................kmeans__n_clusters=99; total time=   0.0s


/home/haider/anaconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/haider/anaconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules

,estimator,Pipeline(step...egression())])
,param_grid,"{'kmeans__n_clusters': range(2, 100)}"
,scoring,None
,n_jobs,None
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_clusters,40


In [22]:
grid_clf.best_params_

{'kmeans__n_clusters': 40}

In [23]:
grid_clf.score(X_test, y_test)

0.9555555555555556